#Initializations

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

#Read Bronze Layer

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

#Silver Transformations

## Trimming leading and trailing spaces

In [0]:
#Instead of reconstructing the DataFrame plan for each column using a for loop, this method updates all string columns in a single plan, making it faster.
#.alias(file.name) - keeps the origianl name of the column instead of "trim(prd_id)" for example

# Old Code:
# for field in df.schema.fields:
#     if isinstance(field.dataType, StringType):
#         df = df.withColumn(field.name, trim(col(field.name)))


df = df.select([
    trim(col(field.name)).alias(field.name) if isinstance(field.dataType, StringType) 
    else col(field.name) 
    for field in df.schema.fields
])

#Customer ID Cleanup

In [0]:

df = df.withColumn(
    "cid",
    F.when(
    col("cid").startswith("NAS"),
    F.substring(
    col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

#Birthday Validation 

In [0]:

df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

#Gender Normalization

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)
     

#Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Sanity checks of dataframe

In [0]:
df.limit(10).display

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

#Sanity Checks of Silver Table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers